<a href="https://colab.research.google.com/github/DonnyGit-08/solesearch-ai/blob/main/SoleSearch_01_prepare_dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SoleSearch AI — Step 1: Prepare the dataset

This notebook downloads the **Popular Sneakers Classification** dataset from Kaggle, keeps only our 10 sneaker models, resizes the images to 512 px, and saves them to Google Drive in `SoleSearch/train_data/<class>/`.

Everything happens in Colab and Drive, so nothing is stored on your laptop. Run the cells top to bottom.

## 1. Connect Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT = Path('/content/drive/MyDrive/SoleSearch')
TRAIN_DIR = PROJECT / 'train_data'
TRAIN_DIR.mkdir(parents=True, exist_ok=True)

MIN_IMAGES = 100  # target number of images per sneaker model
print('Images will be saved to:', TRAIN_DIR)

Mounted at /content/drive
Images will be saved to: /content/drive/MyDrive/SoleSearch/train_data


## 2. Download the Kaggle dataset

Public datasets usually download without logging in. If this cell gives an authentication error:
1. On kaggle.com go to **Settings → API → Create New Token** (downloads `kaggle.json` with your username and key).
2. In Colab, click the **key icon** in the left sidebar (Secrets) and add `KAGGLE_USERNAME` and `KAGGLE_KEY`, with notebook access turned on.
3. Re-run this cell.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception:
    pass  # no secrets set, which is fine for public datasets

import kagglehub
KAGGLE_DIR = Path(kagglehub.dataset_download('nikolasgegenava/sneakers-classification'))
print('Downloaded to:', KAGGLE_DIR)

100%|██████████| 17.1M/17.1M [00:02<00:00, 8.65MB/s]

Extracting files...


Downloaded to: /root/.cache/kagglehub/datasets/nikolasgegenava/sneakers-classification/versions/2


## 3. See every class in the dataset

This lists every folder of images and how many images each has. If the dataset is split into train/test folders, images with the same class name are combined (we make our own split when training).

In [ ]:
import re
from collections import defaultdict

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}

def normalize(name):
    """'Nike_Air-Jordan_1 High' -> 'nike air jordan 1 high'"""
    return re.sub(r'[^a-z0-9]+', ' ', name.lower()).strip()

source_classes = defaultdict(list)  # folder name -> list of image paths
for folder in KAGGLE_DIR.rglob('*'):
    if folder.is_dir():
        images = [p for p in folder.iterdir() if p.suffix.lower() in IMG_EXTS]
        if images:
            source_classes[folder.name].extend(images)

print(f'Found {len(source_classes)} classes:\n')
for name in sorted(source_classes):
    print(f'{len(source_classes[name]):5d}  {name}')

stats_files = list(KAGGLE_DIR.rglob('dataset_stats.csv'))
if stats_files:
    import pandas as pd
    display(pd.read_csv(stats_files[0]))

Found 50 classes:

  150  adidas_forum_high
   92  adidas_forum_low
  149  adidas_gazelle
   95  adidas_nmd_r1
   74  adidas_samba
  147  adidas_stan_smith
   93  adidas_superstar
  150  adidas_ultraboost
   91  asics_gel-lyte_iii
   75  converse_chuck_70_high
  148  converse_chuck_70_low
   78  converse_chuck_taylor_all-star_high
   93  converse_chuck_taylor_all-star_low
  150  converse_one_star
  108  new_balance_327
  150  new_balance_550
  150  new_balance_574
   95  new_balance_990
  150  new_balance_992
   87  nike_air_force_1_high
  147  nike_air_force_1_low
  148  nike_air_force_1_mid
   93  nike_air_jordan_11
   77  nike_air_jordan_1_high
   79  nike_air_jordan_1_low
  100  nike_air_jordan_3
  150  nike_air_jordan_4
  106  nike_air_max_1
  149  nike_air_max_270
  150  nike_air_max_90
   86  nike_air_max_95
   97  nike_air_max_97
   93  nike_air_max_plus_(tn)
  149  nike_air_vapormax_flyknit
  107  nike_air_vapormax_plus
   97  nike_blazer_mid_77
  150  nike_cortez
  150  nike_

,class,image_count,avg_width,avg_height,min_width,min_height,max_width,max_height,formats,corrupt_files
0,adidas_forum_high,150,143,124,78,81,162,140,jpeg,0
1,adidas_ultraboost,150,142,128,93,49,162,140,jpeg,0
2,new_balance_550,150,134,129,79,40,162,140,jpeg,0
3,new_balance_574,150,131,133,78,68,162,140,jpeg,0
4,converse_one_star,150,138,130,82,67,162,140,jpeg,0
5,nike_cortez,150,137,131,93,76,162,140,jpeg,0
6,nike_dunk_high,150,147,122,93,56,162,140,jpeg,0
7,nike_air_max_90,150,141,127,84,65,162,140,jpeg,0
8,nike_air_jordan_4,150,137,130,93,84,162,140,jpeg,0
9,new_balance_992,150,144,122,76,73,162,140,jpeg,0


## 4. Match Kaggle classes to our 10 sneaker models

Each of our classes has search patterns that are checked against the Kaggle folder names. **Read the printout before copying.** If a match is wrong or a folder was missed, edit `CLASS_RULES` or add a folder name to `EXCLUDE`, then re-run this cell.

Decisions built in (explain these in your report):
- **Jordan 1 High and Low are merged** into one `air_jordan_1` class. The colorway matcher handles the finer detail.
- **Air Force 1 Low/Mid/High are merged** into `nike_air_force_1`.
- **Dunk High is not included**, since only Dunk Low is in scope.
- SB Dunk is checked before Dunk Low, so SB folders don't land in the Dunk Low class.

In [ ]:
# Our class name -> regex patterns. The first class that matches wins.
CLASS_RULES = {
    'nike_sb_dunk':     [r'\bsb\b.*\bdunk\b', r'\bdunk\b.*\bsb\b'],
    'nike_dunk_low':    [r'\bdunk low\b'],
    'air_jordan_1':     [r'\bjordan 1\b'],  # \b stops this from matching Jordan 11
    'air_jordan_4':     [r'\bjordan 4\b'],
    'nike_air_force_1': [r'\bair force 1\b', r'\baf1\b'],
    'yeezy_boost_350':  [r'\byeezy\b.*\b350\b', r'\b350\b.*\bv2\b'],
    'adidas_samba':     [r'\bsamba\b'],
    'new_balance_550':  [r'\b550\b'],
    'new_balance_9060': [r'\b9060\b'],
    'asics_gel_kayano': [r'\bkayano\b'],
}

EXCLUDE = set()  # e.g. {'some_folder_name'} to skip a folder that matched by mistake

def match_class(folder_name):
    n = normalize(folder_name)
    for cls, patterns in CLASS_RULES.items():
        if any(re.search(p, n) for p in patterns):
            return cls
    return None

mapping = defaultdict(list)  # our class -> Kaggle folder names
for name in sorted(source_classes):
    cls = match_class(name)
    if cls and name not in EXCLUDE:
        mapping[cls].append(name)

print('Check these matches before copying:\n')
for cls in CLASS_RULES:
    found = mapping.get(cls, [])
    n = sum(len(source_classes[f]) for f in found)
    print(f'{cls:18s} {n:5d} images  <- {", ".join(found) if found else "(none found)"}')

Check these matches before copying:

nike_sb_dunk           0 images  <- (none found)
nike_dunk_low         80 images  <- nike_dunk_low
air_jordan_1         156 images  <- nike_air_jordan_1_high, nike_air_jordan_1_low
air_jordan_4         150 images  <- nike_air_jordan_4
nike_air_force_1     382 images  <- nike_air_force_1_high, nike_air_force_1_low, nike_air_force_1_mid
yeezy_boost_350      148 images  <- yeezy_boost_350_v2
adidas_samba          74 images  <- adidas_samba
new_balance_550      150 images  <- new_balance_550
new_balance_9060       0 images  <- (none found)
asics_gel_kayano       0 images  <- (none found)


## 5. Copy, resize, and save to Drive

Images are resized so the longest side is 512 px and saved as JPEG. Files are named `kaggle_<source>_<id>.jpg`, so you can always tell which images came from Kaggle versus ones you collected. Re-running skips images that are already saved.

Writing a few thousand files to Drive can take 5–15 minutes.

In [ ]:
import hashlib
from PIL import Image
from tqdm.auto import tqdm

def save_resized(src, dst, max_side=512):
    if dst.exists():
        return 'skipped'
    try:
        img = Image.open(src).convert('RGB')
        img.thumbnail((max_side, max_side))  # keeps the aspect ratio
        dst.parent.mkdir(parents=True, exist_ok=True)
        img.save(dst, 'JPEG', quality=90)
        return 'saved'
    except Exception:
        return 'broken'

results = defaultdict(lambda: defaultdict(int))
for cls, folders in mapping.items():
    for folder in folders:
        tag = normalize(folder).replace(' ', '-')
        for src in tqdm(source_classes[folder], desc=f'{cls} <- {folder}'):
            uid = hashlib.md5(str(src).encode()).hexdigest()[:10]  # unique even across train/test splits
            dst = TRAIN_DIR / cls / f'kaggle_{tag}_{uid}.jpg'
            results[cls][save_resized(src, dst)] += 1

for cls, r in results.items():
    print(f'{cls:18s} saved {r["saved"]}, skipped {r["skipped"]}, broken {r["broken"]}')

adidas_samba <- adidas_samba:   0%|          | 0/74 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


new_balance_550 <- new_balance_550:   0%|          | 0/150 [00:00<?, ?it/s]

nike_air_force_1 <- nike_air_force_1_high:   0%|          | 0/87 [00:00<?, ?it/s]

nike_air_force_1 <- nike_air_force_1_low:   0%|          | 0/147 [00:00<?, ?it/s]

nike_air_force_1 <- nike_air_force_1_mid:   0%|          | 0/148 [00:00<?, ?it/s]

... (2 outputs left out)

air_jordan_4 <- nike_air_jordan_4:   0%|          | 0/150 [00:00<?, ?it/s]

nike_dunk_low <- nike_dunk_low:   0%|          | 0/80 [00:00<?, ?it/s]

yeezy_boost_350 <- yeezy_boost_350_v2:   0%|          | 0/148 [00:00<?, ?it/s]

adidas_samba       saved 74, skipped 0, broken 0
new_balance_550    saved 150, skipped 0, broken 0
nike_air_force_1   saved 382, skipped 0, broken 0
air_jordan_1       saved 156, skipped 0, broken 0
air_jordan_4       saved 150, skipped 0, broken 0
nike_dunk_low      saved 80, skipped 0, broken 0
yeezy_boost_350    saved 148, skipped 0, broken 0


## 6. What's still missing?

This counts everything in `train_data`, including photos you add by hand later, so re-run it any time to check progress.

In [ ]:
print(f'{"class":18s} {"images":>6s}  status')
print('-' * 50)
for cls in CLASS_RULES:
    folder = TRAIN_DIR / cls
    n = len(list(folder.glob('*.jpg'))) if folder.exists() else 0
    if n >= MIN_IMAGES:
        status = 'OK'
    elif n == 0:
        status = 'MISSING - needs another source'
    else:
        status = f'LOW - add about {MIN_IMAGES - n} more'
    print(f'{cls:18s} {n:6d}  {status}')

class              images  status
--------------------------------------------------
nike_sb_dunk            0  MISSING - needs another source
nike_dunk_low          80  LOW - add about 20 more
air_jordan_1          156  OK
air_jordan_4          150  OK
nike_air_force_1      382  OK
yeezy_boost_350       148  OK
adidas_samba           74  LOW - add about 26 more
new_balance_550       150  OK
new_balance_9060        0  MISSING - needs another source
asics_gel_kayano        0  MISSING - needs another source
